In [0]:
from pyspark.sql.functions import sum as _sum, count, avg, round, col

# 1. Base Paths Setup
silver_base_path = "abfss://silver@staecommercegen2.dfs.core.windows.net/"
gold_base_path   = "abfss://gold@staecommercegen2.dfs.core.windows.net/"

print("Starting Gold Layer Processing...\n")

# 2. Load Silver Layer Tables
df_users   = spark.read.format("delta").load(f"{silver_base_path}dim_users")
df_products = spark.read.format("delta").load(f"{silver_base_path}dim_products")
df_orders   = spark.read.format("delta").load(f"{silver_base_path}fact_orders")

# -------------------------------------------------------------------------
# A. CUSTOMER SALES SUMMARY (Gold Table)
# -------------------------------------------------------------------------
print("Creating Customer Sales Summary Table...")
gold_customer_summary = df_orders.join(df_users, "user_id", "inner") \
    .groupBy("user_id", "first_name", "last_name", "email") \
    .agg(
        count("cart_id").alias("total_orders"),
        round(_sum("item_discounted_price"), 2).alias("total_spent_amount"),
        round(avg("item_discounted_price"), 2).alias("avg_order_value")
    )

gold_customer_summary.write.format("delta").mode("overwrite").save(f"{gold_base_path}gold_customer_sales_summary")
print("✅ Saved gold_customer_sales_summary!\n")

# -------------------------------------------------------------------------
# B. PRODUCT PERFORMANCE SUMMARY (Gold Table)
# -------------------------------------------------------------------------
print("Creating Product Performance Table...")
gold_product_performance = df_orders.join(df_products, "product_id", "inner") \
    .groupBy("product_id", "product_name", "category", "brand") \
    .agg(
        _sum("quantity").alias("total_quantity_sold"),
        round(_sum("item_discounted_price"), 2).alias("total_revenue_generated"),
        round(avg("discount_percentage"), 2).alias("avg_discount_given")
    )

gold_product_performance.write.format("delta").mode("overwrite").save(f"{gold_base_path}gold_product_performance")
print("✅ Saved gold_product_performance!\n")

print("🚀 Gold Layer Processing Complete!")

Starting Gold Layer Processing...

Creating Customer Sales Summary Table...
✅ Saved gold_customer_sales_summary!

Creating Product Performance Table...
✅ Saved gold_product_performance!

🚀 Gold Layer Processing Complete!


In [0]:
df_gold_customers = spark.read.format("delta").load("abfss://gold@staecommercegen2.dfs.core.windows.net/gold_customer_sales_summary")
display(df_gold_customers)

user_id,first_name,last_name,email,total_orders,total_spent_amount,avg_order_value
4,James,Davis,james.davis@x.dummyjson.com,2,636.81,318.41
24,Mateo,Nguyen,mateo.nguyen@x.dummyjson.com,2,26780.04,13390.02
26,Evelyn,Gonzalez,evelyn.gonzalez@x.dummyjson.com,4,1021.64,255.41
23,Chloe,Morales,chloe.morales@x.dummyjson.com,2,2123.87,1061.94
27,Daniel,Cook,daniel.cook@x.dummyjson.com,4,100496.36,25124.09
30,Addison,Wright,addison.wright@x.dummyjson.com,6,155507.16,25917.86
14,Charlotte,Lopez,charlotte.lopez@x.dummyjson.com,3,184.59,61.53
6,Olivia,Wilson,olivia.wilson@x.dummyjson.com,4,46641.06,11660.27
18,Logan,Torres,logan.torres@x.dummyjson.com,2,124978.81,62489.41
19,Abigail,Rivera,abigail.rivera@x.dummyjson.com,5,21308.13,4261.63


In [0]:
# 1. Unity Catalog mein Schema/Database create karo
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce_gold")

# 2. Customer Sales Summary Table register karo
spark.sql("""
CREATE TABLE IF NOT EXISTS ecommerce_gold.customer_sales_summary
USING DELTA
LOCATION 'abfss://gold@staecommercegen2.dfs.core.windows.net/gold_customer_sales_summary'
""")

# 3. Product Performance Table register karo
spark.sql("""
CREATE TABLE IF NOT EXISTS ecommerce_gold.product_performance
USING DELTA
LOCATION 'abfss://gold@staecommercegen2.dfs.core.windows.net/gold_product_performance'
""")

print("✅ Unity Catalog Tables Registered Successfully!")

✅ Unity Catalog Tables Registered Successfully!


In [0]:
# Base Gold External Location path
gold_base_path = "abfss://gold@staecommercegen2.dfs.core.windows.net/"

print("--- Root of 'gold' container ---")
root_items = dbutils.fs.ls(gold_base_path)
for item in root_items:
    print(f"Folder/File: {item.path}")

print("\n--- Checking _delta_log inside gold_customer_sales_summary ---")
try:
    cust_logs = dbutils.fs.ls(f"{gold_base_path}gold_customer_sales_summary/")
    for item in cust_logs:
        print(f"  -> {item.name}")
except Exception as e:
    print(f"Error checking customer sales summary: {e}")

print("\n--- Checking _delta_log inside gold_product_performance ---")
try:
    prod_logs = dbutils.fs.ls(f"{gold_base_path}gold_product_performance/")
    for item in prod_logs:
        print(f"  -> {item.name}")
except Exception as e:
    print(f"Error checking product performance: {e}")

--- Root of 'gold' container ---
Folder/File: abfss://gold@staecommercegen2.dfs.core.windows.net/gold_customer_sales_summary/
Folder/File: abfss://gold@staecommercegen2.dfs.core.windows.net/gold_product_performance/

--- Checking _delta_log inside gold_customer_sales_summary ---
  -> _delta_log/
  -> part-00000-1be8fc8d-ea5d-46c3-9a82-2cc4ca9a179c.c000.snappy.parquet
  -> part-00000-9d510512-654d-4eb1-846c-219334c99595.c000.snappy.parquet

--- Checking _delta_log inside gold_product_performance ---
  -> _delta_log/
  -> part-00000-0a5d84fc-ca8d-4f42-9b1e-ee1a9b09ef35.c000.snappy.parquet
  -> part-00000-df3c66b3-d01c-41da-8478-e029936329d3.c000.snappy.parquet
